In [1]:
# ============================================================
# Librerías y configuración global
# ============================================================

from pathlib import Path
import os
import sys
import random

import numpy as np
import pandas as pd

import torch
import torchvision
import sklearn


SEED = 42

# Evita entrenamientos costosos durante Restart Kernel + Run All
RUN_TRAINING = False

GENERAR_ARTEFACTOS_A5 = False

GENERAR_MODELOS_A5 = False


def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


fijar_semilla(SEED)

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

In [2]:
# ============================================================
# Localización de la raíz del repositorio
# ============================================================

MARCADOR_REPOSITORIO = (
    Path("Datos")
    / "Preparacion_Datasets"
    / "avance4_split_ground_truth_v2.csv"
)


def find_repo_root():
    candidatos = []

    repo_env = os.environ.get("MNA_REPO_DIR")

    if repo_env:
        candidatos.append(Path(repo_env))

    candidatos.extend(
        [
            Path.cwd(),
            *Path.cwd().parents
        ]
    )

    for candidato in candidatos:
        if (candidato / MARCADOR_REPOSITORIO).is_file():
            return candidato.resolve()

    raise RuntimeError(
        "No se pudo localizar la raíz del repositorio. "
        "Ejecute el notebook desde el repositorio o "
        "defina la variable MNA_REPO_DIR."
    )


REPO_DIR = find_repo_root()

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Repositorio localizado correctamente")

Repositorio localizado correctamente


In [3]:
import numpy as np
import pandas as pd

from sklearn.metrics import average_precision_score

SEED = 42
N_BOOTSTRAP = 2000

baseline_path = (
    REPO_DIR
    / "Notebooks"
    / "checkpoints"
    / "r3d18_baseline_test_predictions.csv"
)

final_path = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
    / "avance5_artefactos"
    / "predicciones_r2plus1d_h5_test.csv"
)

df_baseline = pd.read_csv(baseline_path)
df_final = pd.read_csv(final_path)

# Conservar exactamente los clips presentes en test v2.
df_pareado = (
    df_final[
        [
            "clip_id",
            "grupo_experimental",
            "label",
            "p_r2plus1d_h5"
        ]
    ]
    .merge(
        df_baseline[
            [
                "clip_id",
                "clase_binaria",
                "p_ocultamiento"
            ]
        ],
        on="clip_id",
        how="inner",
        validate="one_to_one"
    )
)

assert len(df_pareado) == 136

assert np.array_equal(
    df_pareado["label"].to_numpy(),
    df_pareado["clase_binaria"].to_numpy()
)

y = df_pareado["label"].to_numpy(dtype=int)

p_baseline = df_pareado[
    "p_ocultamiento"
].to_numpy(dtype=float)

p_final = df_pareado[
    "p_r2plus1d_h5"
].to_numpy(dtype=float)

grupos = df_pareado[
    "grupo_experimental"
].astype(str).to_numpy()

grupos_unicos = np.unique(grupos)

ap_baseline_obs = average_precision_score(
    y,
    p_baseline
)

ap_final_obs = average_precision_score(
    y,
    p_final
)

delta_obs = ap_final_obs - ap_baseline_obs

rng = np.random.default_rng(SEED)

bootstrap_baseline = []
bootstrap_final = []
bootstrap_delta = []

for _ in range(N_BOOTSTRAP):

    muestra_grupos = rng.choice(
        grupos_unicos,
        size=len(grupos_unicos),
        replace=True
    )

    indices = np.concatenate([
        np.where(grupos == grupo)[0]
        for grupo in muestra_grupos
    ])

    y_b = y[indices]

    if y_b.sum() == 0:
        continue

    ap_b = average_precision_score(
        y_b,
        p_baseline[indices]
    )

    ap_f = average_precision_score(
        y_b,
        p_final[indices]
    )

    bootstrap_baseline.append(ap_b)
    bootstrap_final.append(ap_f)
    bootstrap_delta.append(ap_f - ap_b)

ic_baseline = np.percentile(
    bootstrap_baseline,
    [2.5, 97.5]
)

ic_final = np.percentile(
    bootstrap_final,
    [2.5, 97.5]
)

ic_delta = np.percentile(
    bootstrap_delta,
    [2.5, 97.5]
)

print("AP baseline:", round(ap_baseline_obs, 4))
print("IC 95% baseline:", np.round(ic_baseline, 4))

print("AP final:", round(ap_final_obs, 4))
print("IC 95% final:", np.round(ic_final, 4))

print("Delta AP:", round(delta_obs, 4))
print("IC 95% delta:", np.round(ic_delta, 4))

AP baseline: 0.6067
IC 95% baseline: [0.4224 0.9487]
AP final: 0.8041
IC 95% final: [0.3734 0.965 ]
Delta AP: 0.1974
IC 95% delta: [-0.1143  0.228 ]


In [4]:
# ============================================================
# A6 - Dependencias autocontenidas para inferencia RGB
#
# Esta celda permite ejecutar Avance 6 sin haber ejecutado
# Avance 5 en el mismo kernel.
#
# Se reutilizan únicamente artefactos congelados del proyecto:
# - checkpoint R(2+1)D-18/H5
# - predicciones finales de test v2
# - dataset/split persistido
#
# No se entrena ningún modelo.
# ============================================================

from pathlib import Path
import os
import cv2

import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torchvision.models import video as video_models


# ------------------------------------------------------------
# 1. Dispositivo
# ------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Dispositivo A6:", DEVICE)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# 2. Artefactos congelados utilizados por A6
# ------------------------------------------------------------

CHECKPOINT_R2PLUS1D_H5_PATH = (
    REPO_DIR
    / "Notebooks"
    / "checkpoints"
    / "r2plus1d_18_avance4_h5.pt"
)

ARTEFACTOS_A5_DIR = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
    / "avance5_artefactos"
)

PREDICCIONES_R2_TEST_PATH = (
    ARTEFACTOS_A5_DIR
    / "predicciones_r2plus1d_h5_test.csv"
)

THRESHOLD_R2PLUS1D_H5 = 0.99690056


artefactos_requeridos = {
    "checkpoint R(2+1)D-18/H5":
        CHECKPOINT_R2PLUS1D_H5_PATH,

    "predicciones R(2+1)D-18/H5 test v2":
        PREDICCIONES_R2_TEST_PATH,
}


for nombre, ruta in artefactos_requeridos.items():

    if not ruta.is_file():
        raise FileNotFoundError(
            f"No se encontró {nombre}: {ruta}"
        )

    print(
        f"{nombre}: OK"
    )


print(
    "Threshold congelado R(2+1)D-18/H5:",
    THRESHOLD_R2PLUS1D_H5
)


# ------------------------------------------------------------
# 3. Configuración de clips
# ------------------------------------------------------------

NUM_FRAMES = 16
BATCH_INFERENCIA_A6 = 16


def cargar_frames_rgb(
    video_path,
    num_frames=NUM_FRAMES
):
    """
    Lee num_frames distribuidos uniformemente entre
    el primer y último frame del clip.

    Reproduce la preparación utilizada en los avances
    experimentales previos.
    """

    cap = cv2.VideoCapture(
        str(video_path)
    )

    if not cap.isOpened():
        raise RuntimeError(
            "No se pudo abrir el clip: "
            f"{video_path.name}"
        )

    n_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    if n_frames < num_frames:

        cap.release()

        raise RuntimeError(
            f"El clip {video_path.name} "
            f"contiene {n_frames} frames "
            f"y requiere al menos "
            f"{num_frames}."
        )

    indices = np.round(
        np.linspace(
            0,
            n_frames - 1,
            num_frames
        )
    ).astype(int)

    frames = []

    for idx in indices:

        cap.set(
            cv2.CAP_PROP_POS_FRAMES,
            int(idx)
        )

        ret, frame = cap.read()

        if not ret:

            cap.release()

            raise RuntimeError(
                "No se pudo leer el frame "
                f"{idx} del clip "
                f"{video_path.name}"
            )

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        frames.append(
            torch
            .from_numpy(
                frame.copy()
            )
            .permute(
                2,
                0,
                1
            )
        )

    cap.release()

    return torch.stack(
        frames,
        dim=0
    )


# ------------------------------------------------------------
# 4. Dataset RGB local de A6
# ------------------------------------------------------------

class VideoDatasetRGB(Dataset):

    def __init__(
        self,
        dataframe,
        clips_dir,
        transform
    ):
        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.clips_dir = Path(
            clips_dir
        )

        self.transform = transform


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):

        fila = self.df.iloc[idx]

        video_path = (
            self.clips_dir
            / fila["nombre_clip"]
        )

        if not video_path.exists():

            raise FileNotFoundError(
                "No se encontró el clip "
                f"requerido: "
                f"{video_path.name}"
            )

        clip = cargar_frames_rgb(
            video_path
        )

        if self.transform is not None:

            clip = self.transform(
                clip
            )

        etiqueta = torch.tensor(
            int(
                fila["clase_binaria"]
            ),
            dtype=torch.long
        )

        return clip, etiqueta


# ------------------------------------------------------------
# 5. Inferencia local
# ------------------------------------------------------------

@torch.inference_mode()
def inferir_probabilidades_rgb(
    modelo,
    loader,
    device,
    usar_amp=True
):
    """
    Ejecuta inferencia binaria y devuelve:
      y_true : etiquetas enteras
      y_prob : P(clase=ocultamiento)

    Implementación local para que A6 no dependa
    de Scripts.artefactos_avance5.
    """

    modelo.eval()

    etiquetas = []
    probabilidades = []

    amp_activo = (
        usar_amp
        and device.type == "cuda"
    )

    for clips, labels in loader:

        clips = clips.to(
            device,
            non_blocking=True
        )

        if amp_activo:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = modelo(
                    clips
                )

        else:

            logits = modelo(
                clips
            )

        probs = torch.softmax(
            logits,
            dim=1
        )[:, 1]

        etiquetas.append(
            labels
            .detach()
            .cpu()
            .numpy()
        )

        probabilidades.append(
            probs
            .detach()
            .float()
            .cpu()
            .numpy()
        )

    y_true = np.concatenate(
        etiquetas
    ).astype(
        np.int64
    )

    y_prob = np.concatenate(
        probabilidades
    ).astype(
        np.float64
    )

    if not np.isfinite(
        y_prob
    ).all():

        raise RuntimeError(
            "La inferencia produjo "
            "probabilidades no finitas."
        )

    return {
        "y_true": y_true,
        "y_prob": y_prob
    }


print(
    "Dependencias autocontenidas "
    "de inferencia A6: OK"
)

Dispositivo A6: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
checkpoint R(2+1)D-18/H5: OK
predicciones R(2+1)D-18/H5 test v2: OK
Threshold congelado R(2+1)D-18/H5: 0.99690056
Dependencias autocontenidas de inferencia A6: OK


In [5]:
# ============================================================
# A6 - ANÁLISIS DE SENSIBILIDAD DEL GROUND TRUTH
# Test v1 original (147 clips) vs test v2 (136 clips)
#
# Objetivo:
# 1. Puntuar R(2+1)D-18/H5 sobre los 11 positivos ambiguos
#    retirados de test v2.
# 2. Reconstruir las predicciones del candidato sobre test v1.
# 3. Comparar baseline y candidato sobre exactamente los
#    mismos 147 clips.
# 4. Calcular bootstrap pareado por grupo para Delta AP.
#
# No se reentrena ningún modelo.
# No se modifica ningún threshold.
# ============================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision.models import video as video_models

from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)


# ------------------------------------------------------------
# 1. Validaciones previas
# ------------------------------------------------------------

variables_requeridas = [
    "REPO_DIR",
    "DEVICE",
    "CHECKPOINT_R2PLUS1D_H5_PATH",
    "PREDICCIONES_R2_TEST_PATH",
    "THRESHOLD_R2PLUS1D_H5",
    "VideoDatasetRGB",
    "inferir_probabilidades_rgb",
]

faltantes = [
    nombre
    for nombre in variables_requeridas
    if nombre not in globals()
]

if faltantes:
    raise RuntimeError(
        "Ejecute primero las celdas de configuración "
        "de Avance 6. "
        f"Variables faltantes: {faltantes}"
    )

AMBIGUOUS_TEST_CLIPS_A6 = [
    # DCSASS
    "Shoplifting006_x264_28",
    "Shoplifting006_x264_29",
    "Shoplifting006_x264_19",
    "Shoplifting021_x264_23",

    # MNNIT
    "MNNIT_065",
    "MNNIT_067",
    "MNNIT_013",
    "MNNIT_064",
    "MNNIT_069",
    "MNNIT_066",
    "MNNIT_068",
]

THRESHOLD_BASELINE = 0.7090
THRESHOLD_CANDIDATO = float(THRESHOLD_R2PLUS1D_H5)

assert np.isclose(
    THRESHOLD_CANDIDATO,
    0.99690056,
    atol=1e-6
)


# ------------------------------------------------------------
# 2. Recuperar test v1 original y predicciones del baseline
# ------------------------------------------------------------

SPLIT_V2_PATH = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
    / "avance4_split_ground_truth_v2.csv"
)

BASELINE_PRED_PATH = (
    REPO_DIR
    / "Notebooks"
    / "checkpoints"
    / "r3d18_baseline_test_predictions.csv"
)

if not SPLIT_V2_PATH.is_file():
    raise FileNotFoundError(SPLIT_V2_PATH)

if not BASELINE_PRED_PATH.is_file():
    raise FileNotFoundError(BASELINE_PRED_PATH)


df_split_completo = pd.read_csv(SPLIT_V2_PATH)

# El archivo v2 conserva los 985 registros originales;
# los ambiguos están marcados como no elegibles.
df_test_v1_meta = (
    df_split_completo[
        df_split_completo["split"].eq("test")
    ]
    .copy()
    .reset_index(drop=True)
)

df_baseline_v1 = pd.read_csv(BASELINE_PRED_PATH)


assert len(df_test_v1_meta) == 147
assert int(df_test_v1_meta["clase_binaria"].sum()) == 28

assert len(df_baseline_v1) == 147
assert int(df_baseline_v1["clase_binaria"].sum()) == 28


# ------------------------------------------------------------
# 3. Extraer los 11 clips ambiguos
# ------------------------------------------------------------

df_ambiguos = (
    df_test_v1_meta[
        df_test_v1_meta["clip_id"].isin(
            AMBIGUOUS_TEST_CLIPS_A6
        )
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(df_ambiguos) == 11
assert df_ambiguos["clase_binaria"].eq(1).all()


print("=== CLIPS AMBIGUOS ===")
print("Total:", len(df_ambiguos))
print(
    df_ambiguos["dataset"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 4. Preparar solamente los 11 clips para R(2+1)D-18/H5
# ------------------------------------------------------------

ruta_clips = os.environ.get("BNEXT_CLIPS_DIR")

if not ruta_clips:
    raise RuntimeError(
        "BNEXT_CLIPS_DIR no está configurada."
    )

CLIPS_DIR_A6 = Path(ruta_clips)

if not CLIPS_DIR_A6.is_dir():
    raise RuntimeError(
        "BNEXT_CLIPS_DIR no apunta a un directorio válido."
    )


transform_a6 = (
    video_models
    .R2Plus1D_18_Weights
    .DEFAULT
    .transforms()
)

dataset_ambiguos = VideoDatasetRGB(
    df_ambiguos,
    CLIPS_DIR_A6,
    transform_a6
)

loader_ambiguos = DataLoader(
    dataset_ambiguos,
    batch_size=11,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


# ------------------------------------------------------------
# 5. Cargar exactamente el checkpoint H5 actual
# ------------------------------------------------------------

checkpoint_a6 = torch.load(
    CHECKPOINT_R2PLUS1D_H5_PATH,
    map_location="cpu",
    weights_only=False
)

modelo_a6 = video_models.r2plus1d_18(
    weights=None
)

modelo_a6.fc = nn.Linear(
    modelo_a6.fc.in_features,
    2
)

modelo_a6.load_state_dict(
    checkpoint_a6["model_state_dict"]
)

modelo_a6 = (
    modelo_a6
    .to(DEVICE)
    .eval()
)


# ------------------------------------------------------------
# 6. Inferir los 11 ambiguos
# ------------------------------------------------------------

resultado_ambiguos = (
    inferir_probabilidades_rgb(
        modelo_a6,
        loader_ambiguos,
        DEVICE,
        usar_amp=True
    )
)

assert np.array_equal(
    resultado_ambiguos["y_true"],
    df_ambiguos["clase_binaria"].to_numpy(dtype=np.int64)
)

df_pred_ambiguos = df_ambiguos[
    [
        "dataset",
        "video_id",
        "clip_id",
        "grupo_experimental",
        "clase_binaria",
    ]
].copy()

df_pred_ambiguos["p_r2plus1d_h5"] = (
    resultado_ambiguos["y_prob"]
)

df_pred_ambiguos["pred_r2plus1d_h5"] = (
    df_pred_ambiguos["p_r2plus1d_h5"]
    >= THRESHOLD_CANDIDATO
).astype(int)


# ------------------------------------------------------------
# 7. Añadir la probabilidad histórica del baseline
# ------------------------------------------------------------

df_pred_ambiguos = df_pred_ambiguos.merge(
    df_baseline_v1[
        [
            "clip_id",
            "p_ocultamiento",
        ]
    ],
    on="clip_id",
    how="left",
    validate="one_to_one"
)

df_pred_ambiguos = df_pred_ambiguos.rename(
    columns={
        "p_ocultamiento": "p_r3d18_baseline"
    }
)


# ------------------------------------------------------------
# 8. Cargar las 136 predicciones actuales del candidato en v2
# ------------------------------------------------------------

if not PREDICCIONES_R2_TEST_PATH.is_file():
    raise FileNotFoundError(
        PREDICCIONES_R2_TEST_PATH
    )

df_r2_v2 = pd.read_csv(
    PREDICCIONES_R2_TEST_PATH
)

assert len(df_r2_v2) == 136


# ------------------------------------------------------------
# 9. Reconstruir candidato sobre test v1 = 136 + 11
# ------------------------------------------------------------

df_r2_v1 = pd.concat(
    [
        df_r2_v2[
            [
                "clip_id",
                "p_r2plus1d_h5",
            ]
        ],
        df_pred_ambiguos[
            [
                "clip_id",
                "p_r2plus1d_h5",
            ]
        ],
    ],
    ignore_index=True
)

assert len(df_r2_v1) == 147
assert df_r2_v1["clip_id"].nunique() == 147


# ------------------------------------------------------------
# 10. Construir tabla pareada completa de test v1
# ------------------------------------------------------------

df_pareado_v1 = (
    df_test_v1_meta[
        [
            "dataset",
            "video_id",
            "clip_id",
            "grupo_experimental",
            "clase_binaria",
        ]
    ]
    .merge(
        df_baseline_v1[
            [
                "clip_id",
                "p_ocultamiento",
            ]
        ],
        on="clip_id",
        how="inner",
        validate="one_to_one"
    )
    .merge(
        df_r2_v1,
        on="clip_id",
        how="inner",
        validate="one_to_one"
    )
)

assert len(df_pareado_v1) == 147

assert int(
    df_pareado_v1["clase_binaria"].sum()
) == 28


# ------------------------------------------------------------
# 11. Función de métricas dependientes e independientes
# ------------------------------------------------------------

def metricas_sensibilidad(
    y_true,
    y_prob,
    threshold
):
    y_true = np.asarray(y_true, dtype=int)
    y_prob = np.asarray(y_prob, dtype=float)

    y_pred = (
        y_prob >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1]
    ).ravel()

    return {
        "AP": average_precision_score(
            y_true,
            y_prob
        ),
        "precision": precision_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_true,
            y_pred,
            zero_division=0
        ),
        "specificity": (
            tn / (tn + fp)
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


y_v1 = df_pareado_v1[
    "clase_binaria"
].to_numpy(dtype=int)

p_baseline_v1 = df_pareado_v1[
    "p_ocultamiento"
].to_numpy(dtype=float)

p_candidato_v1 = df_pareado_v1[
    "p_r2plus1d_h5"
].to_numpy(dtype=float)


m_baseline_v1 = metricas_sensibilidad(
    y_v1,
    p_baseline_v1,
    THRESHOLD_BASELINE
)

m_candidato_v1 = metricas_sensibilidad(
    y_v1,
    p_candidato_v1,
    THRESHOLD_CANDIDATO
)


# Control: baseline debe reproducir Avance 3
assert np.isclose(
    m_baseline_v1["AP"],
    0.5393,
    atol=5e-5
)


# ------------------------------------------------------------
# 12. Reconstruir también v2 con los mismos vectores
# ------------------------------------------------------------

mask_v2 = ~df_pareado_v1[
    "clip_id"
].isin(
    AMBIGUOUS_TEST_CLIPS_A6
)

df_pareado_v2 = (
    df_pareado_v1[
        mask_v2
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(df_pareado_v2) == 136
assert int(
    df_pareado_v2["clase_binaria"].sum()
) == 17


m_baseline_v2 = metricas_sensibilidad(
    df_pareado_v2["clase_binaria"],
    df_pareado_v2["p_ocultamiento"],
    THRESHOLD_BASELINE
)

m_candidato_v2 = metricas_sensibilidad(
    df_pareado_v2["clase_binaria"],
    df_pareado_v2["p_r2plus1d_h5"],
    THRESHOLD_CANDIDATO
)


assert np.isclose(
    m_baseline_v2["AP"],
    0.6067,
    atol=5e-5
)

assert np.isclose(
    m_candidato_v2["AP"],
    0.8041,
    atol=5e-5
)


# ------------------------------------------------------------
# 13. Comparación v1 vs v2
# ------------------------------------------------------------

tabla_sensibilidad = pd.DataFrame([
    {
        "test": "v1 original",
        "modelo": "R3D-18 baseline",
        **m_baseline_v1
    },
    {
        "test": "v1 original",
        "modelo": "R(2+1)D-18/H5",
        **m_candidato_v1
    },
    {
        "test": "v2 depurado",
        "modelo": "R3D-18 baseline",
        **m_baseline_v2
    },
    {
        "test": "v2 depurado",
        "modelo": "R(2+1)D-18/H5",
        **m_candidato_v2
    },
])


display(
    tabla_sensibilidad.style.format({
        "AP": "{:.4f}",
        "precision": "{:.4f}",
        "recall": "{:.4f}",
        "F1": "{:.4f}",
        "specificity": "{:.4f}",
    })
)


# ------------------------------------------------------------
# 14. Efecto puntual de la auditoría
# ------------------------------------------------------------

delta_ap_v1 = (
    m_candidato_v1["AP"]
    - m_baseline_v1["AP"]
)

delta_ap_v2 = (
    m_candidato_v2["AP"]
    - m_baseline_v2["AP"]
)

n_detectados_ambiguos = int(
    df_pred_ambiguos[
        "pred_r2plus1d_h5"
    ].sum()
)


print("\n=== RESULTADOS PRINCIPALES ===")

print(
    "Ambiguos detectados por R2/H5:",
    f"{n_detectados_ambiguos}/11"
)

print(
    "AP baseline test v1:",
    round(m_baseline_v1["AP"], 4)
)

print(
    "AP candidato test v1:",
    round(m_candidato_v1["AP"], 4)
)

print(
    "Delta AP test v1:",
    round(delta_ap_v1, 4)
)

print(
    "AP baseline test v2:",
    round(m_baseline_v2["AP"], 4)
)

print(
    "AP candidato test v2:",
    round(m_candidato_v2["AP"], 4)
)

print(
    "Delta AP test v2:",
    round(delta_ap_v2, 4)
)


print("\n=== 11 CLIPS AMBIGUOS ===")

display(
    df_pred_ambiguos[
        [
            "dataset",
            "clip_id",
            "grupo_experimental",
            "p_r3d18_baseline",
            "p_r2plus1d_h5",
            "pred_r2plus1d_h5",
        ]
    ]
    .sort_values(
        "p_r2plus1d_h5",
        ascending=False
    )
    .style.format({
        "p_r3d18_baseline": "{:.6f}",
        "p_r2plus1d_h5": "{:.6f}",
    })
)


# ------------------------------------------------------------
# 15. Bootstrap pareado por grupo
# ------------------------------------------------------------

def bootstrap_delta_ap_por_grupo(
    df,
    n_bootstrap=2000,
    seed=42
):
    rng = np.random.default_rng(seed)

    grupos = (
        df["grupo_experimental"]
        .astype(str)
        .to_numpy()
    )

    grupos_unicos = np.unique(grupos)

    y = df[
        "clase_binaria"
    ].to_numpy(dtype=int)

    p_b = df[
        "p_ocultamiento"
    ].to_numpy(dtype=float)

    p_c = df[
        "p_r2plus1d_h5"
    ].to_numpy(dtype=float)

    delta_boot = []
    ap_b_boot = []
    ap_c_boot = []

    n_sin_positivos = 0

    for _ in range(n_bootstrap):

        muestra_grupos = rng.choice(
            grupos_unicos,
            size=len(grupos_unicos),
            replace=True
        )

        indices = np.concatenate([
            np.where(
                grupos == grupo
            )[0]
            for grupo in muestra_grupos
        ])

        y_b = y[indices]

        if y_b.sum() == 0:
            n_sin_positivos += 1
            continue

        ap_b = average_precision_score(
            y_b,
            p_b[indices]
        )

        ap_c = average_precision_score(
            y_b,
            p_c[indices]
        )

        ap_b_boot.append(ap_b)
        ap_c_boot.append(ap_c)
        delta_boot.append(
            ap_c - ap_b
        )

    return {
        "n_total": n_bootstrap,
        "n_validos": len(delta_boot),
        "n_sin_positivos": n_sin_positivos,
        "ic_baseline": np.percentile(
            ap_b_boot,
            [2.5, 97.5]
        ),
        "ic_candidato": np.percentile(
            ap_c_boot,
            [2.5, 97.5]
        ),
        "ic_delta": np.percentile(
            delta_boot,
            [2.5, 97.5]
        ),
    }


bootstrap_v1 = bootstrap_delta_ap_por_grupo(
    df_pareado_v1,
    n_bootstrap=2000,
    seed=42
)

bootstrap_v2 = bootstrap_delta_ap_por_grupo(
    df_pareado_v2,
    n_bootstrap=2000,
    seed=42
)


print("\n=== BOOTSTRAP PAREADO TEST V1 ===")
print(
    "Réplicas válidas:",
    bootstrap_v1["n_validos"],
    "/",
    bootstrap_v1["n_total"]
)
print(
    "Réplicas sin positivos:",
    bootstrap_v1["n_sin_positivos"]
)
print(
    "IC95% AP baseline:",
    np.round(
        bootstrap_v1["ic_baseline"],
        4
    )
)
print(
    "IC95% AP candidato:",
    np.round(
        bootstrap_v1["ic_candidato"],
        4
    )
)
print(
    "IC95% Delta AP:",
    np.round(
        bootstrap_v1["ic_delta"],
        4
    )
)


print("\n=== CONTROL BOOTSTRAP TEST V2 ===")
print(
    "IC95% Delta AP:",
    np.round(
        bootstrap_v2["ic_delta"],
        4
    )
)


# ------------------------------------------------------------
# 16. Guardar evidencia reproducible para Avance 6
# ------------------------------------------------------------

SALIDA_DIR = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
    / "avance6_artefactos"
)

print(
    "Artefactos A6:",
    SALIDA_DIR.relative_to(
        REPO_DIR
    )
)

SALIDA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Artefactos A6:",
    SALIDA_DIR.relative_to(
        REPO_DIR
    )
)

SALIDA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

tabla_sensibilidad.to_csv(
    SALIDA_DIR
    / "sensibilidad_ground_truth_test_v1_v2.csv",
    index=False
)

df_pred_ambiguos.to_csv(
    SALIDA_DIR
    / "sensibilidad_11_ambiguos_r2plus1d_h5.csv",
    index=False
)

pd.DataFrame([
    {
        "dataset": "test_v1",
        "delta_ap": delta_ap_v1,
        "bootstrap_validos": bootstrap_v1["n_validos"],
        "bootstrap_sin_positivos": bootstrap_v1["n_sin_positivos"],
        "ic_delta_inf": bootstrap_v1["ic_delta"][0],
        "ic_delta_sup": bootstrap_v1["ic_delta"][1],
    },
    {
        "dataset": "test_v2",
        "delta_ap": delta_ap_v2,
        "bootstrap_validos": bootstrap_v2["n_validos"],
        "bootstrap_sin_positivos": bootstrap_v2["n_sin_positivos"],
        "ic_delta_inf": bootstrap_v2["ic_delta"][0],
        "ic_delta_sup": bootstrap_v2["ic_delta"][1],
    }
]).to_csv(
    SALIDA_DIR
    / "sensibilidad_ground_truth_bootstrap.csv",
    index=False
)


# Liberar memoria GPU
del modelo_a6
del checkpoint_a6

if torch.cuda.is_available():
    torch.cuda.empty_cache()

=== CLIPS AMBIGUOS ===
Total: 11
dataset
DCSASS    4
MNNIT     7
Name: count, dtype: int64


,test,modelo,AP,precision,recall,F1,specificity,TN,FP,FN,TP
0,v1 original,R3D-18 baseline,0.5393,0.5769,0.5357,0.5556,0.9076,108,11,13,15
1,v1 original,R(2+1)D-18/H5,0.7459,0.8125,0.4643,0.5909,0.9748,116,3,15,13
2,v2 depurado,R3D-18 baseline,0.6067,0.5769,0.8824,0.6977,0.9076,108,11,2,15
3,v2 depurado,R(2+1)D-18/H5,0.8041,0.7857,0.6471,0.7097,0.9748,116,3,6,11



=== RESULTADOS PRINCIPALES ===
Ambiguos detectados por R2/H5: 2/11
AP baseline test v1: 0.5393
AP candidato test v1: 0.7459
Delta AP test v1: 0.2067
AP baseline test v2: 0.6067
AP candidato test v2: 0.8041
Delta AP test v2: 0.1974

=== 11 CLIPS AMBIGUOS ===


,dataset,clip_id,grupo_experimental,p_r3d18_baseline,p_r2plus1d_h5,pred_r2plus1d_h5
5,MNNIT,MNNIT_064,G039,0.384000,0.999512,1
4,MNNIT,MNNIT_013,G039,0.371000,0.999023,1
6,MNNIT,MNNIT_065,G039,0.160900,0.995605,0
8,MNNIT,MNNIT_067,G039,0.334000,0.990234,0
9,MNNIT,MNNIT_068,G039,0.615700,0.987793,0
7,MNNIT,MNNIT_066,G039,0.439200,0.982910,0
10,MNNIT,MNNIT_069,G039,0.414800,0.978027,0
3,DCSASS,Shoplifting021_x264_23,G014,0.009450,0.694824,0
1,DCSASS,Shoplifting006_x264_28,G007,0.037380,0.063660,0
0,DCSASS,Shoplifting006_x264_19,G007,0.126200,0.034576,0



=== BOOTSTRAP PAREADO TEST V1 ===
Réplicas válidas: 2000 / 2000
Réplicas sin positivos: 0
IC95% AP baseline: [0.3081 0.9243]
IC95% AP candidato: [0.2563 0.9176]
IC95% Delta AP: [-0.1102  0.2451]

=== CONTROL BOOTSTRAP TEST V2 ===
IC95% Delta AP: [-0.1143  0.228 ]
Artefactos A6: Datos/Preparacion_Datasets/avance6_artefactos
Artefactos A6: Datos/Preparacion_Datasets/avance6_artefactos


In [6]:
# ============================================================
# A6 - Comparación de curvas a restricciones equivalentes
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

# Usa df_pareado_v2 generado en el análisis de sensibilidad.
y = df_pareado_v2["clase_binaria"].to_numpy(dtype=int)

p_baseline = df_pareado_v2[
    "p_ocultamiento"
].to_numpy(dtype=float)

p_candidato = df_pareado_v2[
    "p_r2plus1d_h5"
].to_numpy(dtype=float)

fpr_b, tpr_b, thr_b = roc_curve(y, p_baseline)
fpr_c, tpr_c, thr_c = roc_curve(y, p_candidato)

recall_baseline_operativo = 15 / 17
fpr_candidato_operativo = 3 / 119


def minimo_fpr_para_recall(fpr, tpr, thresholds, recall_objetivo):
    idx = np.where(tpr >= recall_objetivo)[0]
    if len(idx) == 0:
        return None

    mejor = idx[np.argmin(fpr[idx])]

    return {
        "threshold": float(thresholds[mejor]),
        "recall": float(tpr[mejor]),
        "fpr": float(fpr[mejor]),
    }


def maximo_recall_para_fpr(fpr, tpr, thresholds, fpr_objetivo):
    idx = np.where(fpr <= fpr_objetivo)[0]
    if len(idx) == 0:
        return None

    mejor = idx[np.argmax(tpr[idx])]

    return {
        "threshold": float(thresholds[mejor]),
        "recall": float(tpr[mejor]),
        "fpr": float(fpr[mejor]),
    }


candidato_a_recall_baseline = minimo_fpr_para_recall(
    fpr_c,
    tpr_c,
    thr_c,
    recall_baseline_operativo
)

baseline_a_fpr_candidato = maximo_recall_para_fpr(
    fpr_b,
    tpr_b,
    thr_b,
    fpr_candidato_operativo
)

print("=== Comparación descriptiva sobre test v2 ===")
print(
    "\nCandidato con recall >= recall operativo baseline:"
)
print(candidato_a_recall_baseline)

print(
    "\nBaseline con FPR <= FPR operativo candidato:"
)
print(baseline_a_fpr_candidato)

=== Comparación descriptiva sobre test v2 ===

Candidato con recall >= recall operativo baseline:
{'threshold': 0.9393595, 'recall': 0.9411764705882353, 'fpr': 0.10084033613445378}

Baseline con FPR <= FPR operativo candidato:
{'threshold': 0.9795, 'recall': 0.29411764705882354, 'fpr': 0.025210084033613446}


In [7]:
# ============================================================
# A6 - Composición de test v2 por fuente y grupo
# ============================================================

tabla_fuentes_test_v2 = (
    df_pareado_v2
    .groupby("dataset")
    .agg(
        clips=("clip_id", "size"),
        grupos=("grupo_experimental", "nunique"),
        positivos=("clase_binaria", "sum")
    )
    .reset_index()
)

tabla_fuentes_test_v2["negativos"] = (
    tabla_fuentes_test_v2["clips"]
    - tabla_fuentes_test_v2["positivos"]
)

tabla_fuentes_test_v2["prevalencia"] = (
    tabla_fuentes_test_v2["positivos"]
    / tabla_fuentes_test_v2["clips"]
)

display(tabla_fuentes_test_v2)


tabla_grupos_test_v2 = (
    df_pareado_v2
    .groupby(
        ["dataset", "grupo_experimental"]
    )
    .agg(
        clips=("clip_id", "size"),
        positivos=("clase_binaria", "sum")
    )
    .reset_index()
)

tabla_grupos_test_v2["negativos"] = (
    tabla_grupos_test_v2["clips"]
    - tabla_grupos_test_v2["positivos"]
)

display(tabla_grupos_test_v2)

,dataset,clips,grupos,positivos,negativos,prevalencia
0,CCTV_YOLO,2,1,1,1,0.500000
1,DCSASS,108,4,7,101,0.064815
2,MNNIT,25,2,8,17,0.320000
3,SINTETICO,1,1,1,0,1.000000


,dataset,grupo_experimental,clips,positivos,negativos
0,CCTV_YOLO,G003,2,1,1
1,DCSASS,G007,13,0,13
2,DCSASS,G014,31,1,30
3,DCSASS,G022,32,0,32
4,DCSASS,G027,32,6,26
5,MNNIT,G036,1,0,1
6,MNNIT,G039,24,8,16
7,SINTETICO,G053,1,1,0


In [8]:
# ============================================================
# A6 - Prevalencia exacta de RetailS_test_staged
#
# Lee exclusivamente archivos NPY binarios frame-level.
# Controles esperados del Avance 4:
#   624 fuentes
#   40,913 frames
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd


RAIZ_RETAILS_BNEXT = (
    Path.home()
    / "mnt"
    / "bnext_retails"
)

RAIZ_STAGED = (
    RAIZ_RETAILS_BNEXT
    / "RetailS_test_staged"
)

if not RAIZ_STAGED.is_dir():
    raise FileNotFoundError(
        f"No existe: {RAIZ_STAGED}"
    )


registros_labels = []


for ruta in sorted(
    RAIZ_STAGED.rglob("*.npy")
):

    try:
        arr = np.load(
            ruta,
            allow_pickle=False
        )
    except Exception:
        continue

    arr = np.asarray(arr).squeeze()

    # Las etiquetas esperadas son vectores frame-level.
    if arr.ndim != 1 or arr.size == 0:
        continue

    valores = np.unique(arr)

    # Conservar únicamente arreglos binarios.
    if not set(
        valores.tolist()
    ).issubset({0, 1}):
        continue

    positivos = int(
        np.sum(arr == 1)
    )

    negativos = int(
        np.sum(arr == 0)
    )

    registros_labels.append(
        {
            "archivo": str(
                ruta.relative_to(
                    RAIZ_STAGED
                )
            ),
            "frames": int(arr.size),
            "positivos": positivos,
            "negativos": negativos,
        }
    )


df_labels_staged = pd.DataFrame(
    registros_labels
)


print(
    "Archivos binarios candidatos:",
    len(df_labels_staged)
)

print(
    "Frames totales:",
    df_labels_staged[
        "frames"
    ].sum()
)


# Controles contra Avance 4.
assert len(df_labels_staged) == 624, (
    "Se esperaban 624 archivos/fuentes "
    "de etiquetas. Revisar selección de NPY."
)

assert (
    df_labels_staged["frames"].sum()
    == 40913
), (
    "El total de frames no reproduce "
    "los 40,913 reportados en Avance 4."
)


positivos_staged = int(
    df_labels_staged[
        "positivos"
    ].sum()
)

negativos_staged = int(
    df_labels_staged[
        "negativos"
    ].sum()
)

total_staged = (
    positivos_staged
    + negativos_staged
)

prevalencia_staged = (
    positivos_staged
    / total_staged
)


print(
    "\n=== RetailS_test_staged ==="
)

print(
    "Fuentes:",
    len(df_labels_staged)
)

print(
    "Frames:",
    total_staged
)

print(
    "Positivos:",
    positivos_staged
)

print(
    "Negativos:",
    negativos_staged
)

print(
    "Prevalencia positiva:",
    f"{prevalencia_staged:.6f}"
)

print(
    "AP STG-NF:",
    "0.7130"
)

print(
    "Margen AP sobre prevalencia:",
    f"{0.7130 - prevalencia_staged:.4f}"
)


# Persistir evidencia A6
SALIDA_DIR = (
    REPO_DIR
    / "Datos"
    / "Preparacion_Datasets"
    / "avance6_artefactos"
)

SALIDA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

df_labels_staged.to_csv(
    SALIDA_DIR
    / "retails_staged_composicion_frame.csv",
    index=False
)

pd.DataFrame(
    [{
        "fuentes": len(df_labels_staged),
        "frames": total_staged,
        "positivos": positivos_staged,
        "negativos": negativos_staged,
        "prevalencia": prevalencia_staged,
        "ap_stgnf": 0.7130,
        "margen_ap_vs_prevalencia":
            0.7130 - prevalencia_staged,
    }]
).to_csv(
    SALIDA_DIR
    / "retails_staged_resumen.csv",
    index=False
)

Archivos binarios candidatos: 624
Frames totales: 40913

=== RetailS_test_staged ===
Fuentes: 624
Frames: 40913
Positivos: 20335
Negativos: 20578
Prevalencia positiva: 0.497030
AP STG-NF: 0.7130
Margen AP sobre prevalencia: 0.2160


In [9]:
# ============================================================
# A6 - Caracterización estática del modelo para Edge
#
# Convención:
# - Se cuentan MAC en Conv3d y Linear.
# - 1 MAC = una multiplicación + una suma.
# - Se reportan FLOPs usando 1 MAC = 2 FLOPs.
# - No se cuentan explícitamente BN, ReLU, pooling,
#   residual additions ni operaciones auxiliares.
#
# Entrada:
#   batch=1, C=3, T=16, H=W=112
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torchvision.models import video as video_models


CHECKPOINT_EDGE = (
    REPO_DIR
    / "Notebooks"
    / "checkpoints"
    / "r2plus1d_18_avance4_h5.pt"
)

if not CHECKPOINT_EDGE.is_file():
    raise FileNotFoundError(
        CHECKPOINT_EDGE
    )


# ------------------------------------------------------------
# 1. Reconstruir modelo binario
# ------------------------------------------------------------

modelo_edge = (
    video_models.r2plus1d_18(
        weights=None
    )
)

modelo_edge.fc = nn.Linear(
    modelo_edge.fc.in_features,
    2
)


checkpoint_edge = torch.load(
    CHECKPOINT_EDGE,
    map_location="cpu",
    weights_only=False
)

modelo_edge.load_state_dict(
    checkpoint_edge[
        "model_state_dict"
    ]
)

modelo_edge.eval()


# ------------------------------------------------------------
# 2. Parámetros y tamaño de pesos
# ------------------------------------------------------------

parametros_totales = sum(
    p.numel()
    for p in modelo_edge.parameters()
)

parametros_entrenables_checkpoint = sum(
    p.numel()
    for p in modelo_edge.parameters()
    if p.requires_grad
)

# Nota: después de reconstruir el modelo todos pueden
# aparecer requires_grad=True. Para Edge interesa
# principalmente el total de parámetros almacenados.

peso_fp32_mib = (
    parametros_totales
    * 4
    / 1024**2
)

peso_fp16_mib = (
    parametros_totales
    * 2
    / 1024**2
)

checkpoint_mib = (
    CHECKPOINT_EDGE.stat().st_size
    / 1024**2
)


# ------------------------------------------------------------
# 3. Conteo MAC mediante forward hooks
# ------------------------------------------------------------

conteo_mac = {
    "conv3d": 0,
    "linear": 0,
}


def hook_conv3d(
    modulo,
    entrada,
    salida
):

    # salida:
    # [B, Cout, T, H, W]

    b = salida.shape[0]
    cout = salida.shape[1]
    tout = salida.shape[2]
    hout = salida.shape[3]
    wout = salida.shape[4]

    cin_por_grupo = (
        modulo.in_channels
        // modulo.groups
    )

    kt, kh, kw = (
        modulo.kernel_size
    )

    mac = (
        b
        * cout
        * tout
        * hout
        * wout
        * cin_por_grupo
        * kt
        * kh
        * kw
    )

    conteo_mac[
        "conv3d"
    ] += int(mac)


def hook_linear(
    modulo,
    entrada,
    salida
):

    mac = (
        salida.numel()
        * modulo.in_features
    )

    conteo_mac[
        "linear"
    ] += int(mac)


handles = []

for modulo in modelo_edge.modules():

    if isinstance(
        modulo,
        nn.Conv3d
    ):
        handles.append(
            modulo.register_forward_hook(
                hook_conv3d
            )
        )

    elif isinstance(
        modulo,
        nn.Linear
    ):
        handles.append(
            modulo.register_forward_hook(
                hook_linear
            )
        )


DEVICE_EDGE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

modelo_edge = modelo_edge.to(
    DEVICE_EDGE
)

entrada_edge = torch.zeros(
    1,
    3,
    16,
    112,
    112,
    device=DEVICE_EDGE
)


with torch.inference_mode():
    _ = modelo_edge(
        entrada_edge
    )


for handle in handles:
    handle.remove()


mac_total = (
    conteo_mac["conv3d"]
    + conteo_mac["linear"]
)

gmac = (
    mac_total
    / 1e9
)

gflops = (
    2
    * mac_total
    / 1e9
)


tabla_edge_estatica = pd.DataFrame(
    [{
        "modelo":
            "R(2+1)D-18/H5",
        "entrada":
            "1x3x16x112x112",
        "parametros_M":
            parametros_totales / 1e6,
        "pesos_FP32_MiB":
            peso_fp32_mib,
        "pesos_FP16_MiB":
            peso_fp16_mib,
        "checkpoint_MiB":
            checkpoint_mib,
        "GMAC_clip":
            gmac,
        "GFLOP_clip_2FLOP_por_MAC":
            gflops,
    }]
)


display(
    tabla_edge_estatica.style.format(
        {
            "parametros_M":
                "{:.3f}",
            "pesos_FP32_MiB":
                "{:.2f}",
            "pesos_FP16_MiB":
                "{:.2f}",
            "checkpoint_MiB":
                "{:.2f}",
            "GMAC_clip":
                "{:.3f}",
            "GFLOP_clip_2FLOP_por_MAC":
                "{:.3f}",
        }
    )
)


print(
    "\nConv3d GMAC:",
    conteo_mac[
        "conv3d"
    ] / 1e9
)

print(
    "Linear GMAC:",
    conteo_mac[
        "linear"
    ] / 1e9
)

print(
    "\nConvención:",
    "1 MAC = 2 FLOPs; "
    "se cuentan Conv3d y Linear."
)

,modelo,entrada,parametros_M,pesos_FP32_MiB,pesos_FP16_MiB,checkpoint_MiB,GMAC_clip,GFLOP_clip_2FLOP_por_MAC
0,R(2+1)D-18/H5,1x3x16x112x112,31.301,119.40,59.70,298.87,40.519,81.038



Conv3d GMAC: 40.51887616
Linear GMAC: 1.024e-06

Convención: 1 MAC = 2 FLOPs; se cuentan Conv3d y Linear.


In [10]:
# ============================================================
# A6 - Benchmark forward-only batch 1
# R(2+1)D-18/H5
#
# Objetivo:
# medir latencia del modelo sin amortización por batch.
#
# Sigue SIN incluir:
# - lectura de video
# - decodificación
# - selección de frames
# - resize / normalización
# - generación de alertas
# ============================================================

import time
import numpy as np
import torch


modelo_edge.eval()

BATCH_EDGE = 1
WARMUP_EDGE = 30
ITERACIONES_EDGE = 200


entrada_b1 = torch.randn(
    BATCH_EDGE,
    3,
    16,
    112,
    112,
    device=DEVICE_EDGE
)


usar_amp_edge = (
    DEVICE_EDGE.type
    == "cuda"
)


# ------------------------------------------------------------
# Warm-up
# ------------------------------------------------------------

with torch.inference_mode():

    for _ in range(
        WARMUP_EDGE
    ):

        if usar_amp_edge:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                _ = modelo_edge(
                    entrada_b1
                )

        else:

            _ = modelo_edge(
                entrada_b1
            )


if torch.cuda.is_available():
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()


# ------------------------------------------------------------
# Medición individual de latencia
# ------------------------------------------------------------

latencias_ms = []


with torch.inference_mode():

    for _ in range(
        ITERACIONES_EDGE
    ):

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        inicio = time.perf_counter()

        if usar_amp_edge:

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                _ = modelo_edge(
                    entrada_b1
                )

        else:

            _ = modelo_edge(
                entrada_b1
            )

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        latencias_ms.append(
            (
                time.perf_counter()
                - inicio
            )
            * 1000
        )


latencias_ms = np.asarray(
    latencias_ms
)


if torch.cuda.is_available():

    memoria_pico_mib = (
        torch.cuda.max_memory_allocated()
        / 1024**2
    )

else:

    memoria_pico_mib = np.nan


resumen_batch1 = pd.DataFrame(
    [{
        "batch": 1,
        "precision":
            (
                "AMP FP16"
                if usar_amp_edge
                else "FP32"
            ),
        "iteraciones":
            ITERACIONES_EDGE,
        "latencia_media_ms":
            latencias_ms.mean(),
        "latencia_mediana_ms":
            np.median(
                latencias_ms
            ),
        "p95_ms":
            np.percentile(
                latencias_ms,
                95
            ),
        "p99_ms":
            np.percentile(
                latencias_ms,
                99
            ),
        "throughput_clips_s":
            1000
            / latencias_ms.mean(),
        "memoria_pico_MiB":
            memoria_pico_mib,
    }]
)


display(
    resumen_batch1.style.format(
        {
            "latencia_media_ms":
                "{:.3f}",
            "latencia_mediana_ms":
                "{:.3f}",
            "p95_ms":
                "{:.3f}",
            "p99_ms":
                "{:.3f}",
            "throughput_clips_s":
                "{:.2f}",
            "memoria_pico_MiB":
                "{:.2f}",
        }
    )
)

,batch,precision,iteraciones,latencia_media_ms,latencia_mediana_ms,p95_ms,p99_ms,throughput_clips_s,memoria_pico_MiB
0,1,AMP FP16,200,27.584,27.543,29.200,30.705,36.25,192.41


In [11]:
# ============================================================
# A6 - Escenarios de ventana / hop por cámara
#
# No son resultados experimentales.
# Son escenarios de diseño para dimensionamiento Edge.
# ============================================================

import pandas as pd


VENTANA_OPERACIONAL_S = 4.0

INFERENCIAS_S_CAMARA = [
    0.5,
    1.0,
    2.0,
]


escenarios_camara = []


for tasa in INFERENCIAS_S_CAMARA:

    hop_s = (
        1.0
        / tasa
    )

    solapamiento = max(
        0.0,
        1.0
        - (
            hop_s
            / VENTANA_OPERACIONAL_S
        )
    )

    escenarios_camara.append(
        {
            "ventana_s":
                VENTANA_OPERACIONAL_S,
            "hop_s":
                hop_s,
            "inferencias_s_camara":
                tasa,
            "inferencias_min_camara":
                tasa * 60,
            "solapamiento_pct":
                100 * solapamiento,
        }
    )


df_escenarios_camara = pd.DataFrame(
    escenarios_camara
)


display(
    df_escenarios_camara.style.format(
        {
            "ventana_s":
                "{:.1f}",
            "hop_s":
                "{:.2f}",
            "inferencias_s_camara":
                "{:.1f}",
            "inferencias_min_camara":
                "{:.0f}",
            "solapamiento_pct":
                "{:.1f}%",
        }
    )
)

,ventana_s,hop_s,inferencias_s_camara,inferencias_min_camara,solapamiento_pct
0,4.0,2.00,0.5,30,50.0%
1,4.0,1.00,1.0,60,75.0%
2,4.0,0.50,2.0,120,87.5%
